In [2]:
#this cell read files and find out it's composition
import pandas as pd
import time
import csv
import re
res=pd.read_csv('zfNGintenstine_20230612_cloud.csv')
res.head(3)

,entry no,MS1scan no,MS1Isolation mass,MS1monoIsomass,chargeState,man_check,in [H+],Structure,MS2 Scan no,peaklist,predictedcomp,hitpeaklist,selectedpeakintensity,normalizedselectedintensity
0,7646,11981,820.431763,820.431763,2,ok,1639.855685,H3N4,12012,"MS2peaklist(dMass=(94.06507110595703, 95.04923...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 344.17, 374.182, 406.203, 4...","[247086.656, 1949057.75, 11567.055, 3674.197, ...","[12.67723626967954, 100.0, 0.5934690749927753,..."
1,8952,14126,820.431946,820.431946,2,ok,1639.856052,H3N4,14168,"MS2peaklist(dMass=(94.06503295898438, 95.04909...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 344.17, 374.18, 432.222, 45...","[19875.078, 1550164.375, 7586.966, 8253.765, 1...","[1.2821271292600827, 100.0, 0.4894297741812057..."
2,7747,12144,820.432129,820.432129,2,ok,1639.856418,H3N4,12180,"MS2peaklist(dMass=(90.99137115478516, 94.06515...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 406.205, 432.222, 450.233, ...","[1047479.938, 6574079.5, 42170.711, 1398368.12...","[15.933484497715611, 100.0, 0.641469440702687,..."


In [5]:
#if the file doesn't include the theoretical value, calculate

#res["peaklist"][1]

def structuretodictparser(dataframe):
    #should be selectable in future
    newdf = dataframe
    exportdf = []
    #handling the data in a df
    newdf.insert(8, 'Structure dict',"")
    for i in range(len(newdf)):
        parser = newdf.iloc[i]['Structure']
        strucdict = {}
        strucdictcom = pd.DataFrame()
        tmpseries = ()
        #start from F
        fuc = re.search('F\d+', parser)
        hexose = re.search('H\d+', parser)
        hexnac = re.search('N\d+', parser)
        neu5ac = re.search('S\d+', parser)
        neu5gc = re.search('G\d+', parser)
        #ambigious strings, get nothing if not present
        Naadduct = re.search('Na', parser)   # if not Naadduct: do nothing ; else add an adduct
        sus = re.search('\?', parser)
        #not included in the first test data
        kdn = re.search('Kdn\d+', parser)
        NH3adduct = re.search('NH3', parser) #may be removed if we calculate this in other function
        Kadduct = re.search('K', parser)
        #conbine the  info into single list in each loop
        if not fuc:
            strucdict.update({'F': 0})
        else:
            strucdict.update({'F': (fuc.group()[1:])})
        if not hexose:
            strucdict.update({'H': 0})
        else:
            strucdict.update({'H': (hexose.group()[1:])})
        if not hexnac:
            strucdict.update({'N': 0})
        else:
            strucdict.update({'N': (hexnac.group()[1:])})
        if not neu5ac:
            strucdict.update({'S': 0})
        else:
            strucdict.update({'S': (neu5ac.group()[1:])})
        if not neu5gc:
            strucdict.update({'G': 0})
        else:
            strucdict.update({'G': (neu5gc.group()[1:])})
        if not kdn:
            strucdict.update({'KDN': 0})
        else:
            strucdict.update({'KDN': (kdn.group()[1:])})
        # I won't add them as default since they are exceptions.
        if not Naadduct:
            pass
            #strucdict.update({'Na': 0) if the loop on the list throws an error, try this
        else:
            strucdict.update({'Na': 1})
        if not sus:
            pass
            #strucdict.update({'Questionable': 0) if the loop on the list throws an error, try this
        else:
            strucdict.update({'Questionable': 1})
        tmpseries = newdf.iloc[i][1:]
        tmpseries['Structure dict'] = strucdict
        exportdf.append(tmpseries)
    exportdf = pd.DataFrame(exportdf)
    exportdf = exportdf.reset_index(drop=True)
    print(exportdf)
    exportdf.to_csv("revised_zfNGintenstine_20230612_cloud.csv", sep='\t')
    #replace the iloc[i]['Structure']  <- SettingWithCopyWarning and the df is not modified.

structuretodictparser(res)

     MS1scan no  MS1Isolation mass  MS1monoIsomass  chargeState   man_check  \
0         11981         820.431763      820.431763            2          ok   
1         14126         820.431946      820.431946            2          ok   
2         12144         820.432129      820.432129            2          ok   
3         12296         820.432312      820.432312            2          ok   
4         14284         820.432617      820.432617            2          ok   
..          ...                ...             ...          ...         ...   
246       31909        1186.604248     1186.268188            3  wrong anno   
247       32264        1186.604492     1186.265137            3  wrong anno   
248       31732        1186.604858     1186.268799            3  wrong anno   
249       30253        1186.618896     1186.273926            3          IF   
250       22265        1794.408447     1793.909302            2   ok (10e5)   

         in [H+]   Structure  \
0    1639.855685   

In [10]:
res2 = pd.read_csv("revised_zfNGintenstine_20230612_cloud.csv", sep ="\t")
res2.reset_index(drop=True)
res2.drop(columns=res2.columns[0], axis=1, inplace=True)
res2.head(5)

,MS1scan no,MS1Isolation mass,MS1monoIsomass,chargeState,man_check,in [H+],Structure,Structure dict,MS2 Scan no,peaklist,predictedcomp,hitpeaklist,selectedpeakintensity,normalizedselectedintensity
0,11981,820.431763,820.431763,2,ok,1639.855685,H3N4,"{'F': 0, 'H': '3', 'N': '4', 'S': 0, 'G': 0, '...",12012,"MS2peaklist(dMass=(94.06507110595703, 95.04923...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 344.17, 374.182, 406.203, 4...","[247086.656, 1949057.75, 11567.055, 3674.197, ...","[12.67723626967954, 100.0, 0.5934690749927753,..."
1,14126,820.431946,820.431946,2,ok,1639.856052,H3N4,"{'F': 0, 'H': '3', 'N': '4', 'S': 0, 'G': 0, '...",14168,"MS2peaklist(dMass=(94.06503295898438, 95.04909...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 344.17, 374.18, 432.222, 45...","[19875.078, 1550164.375, 7586.966, 8253.765, 1...","[1.2821271292600827, 100.0, 0.4894297741812057..."
2,12144,820.432129,820.432129,2,ok,1639.856418,H3N4,"{'F': 0, 'H': '3', 'N': '4', 'S': 0, 'G': 0, '...",12180,"MS2peaklist(dMass=(90.99137115478516, 94.06515...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 406.205, 432.222, 450.233, ...","[1047479.938, 6574079.5, 42170.711, 1398368.12...","[15.933484497715611, 100.0, 0.641469440702687,..."
3,12296,820.432312,820.432312,2,ok,1639.856784,H3N4,"{'F': 0, 'H': '3', 'N': '4', 'S': 0, 'G': 0, '...",12333,"MS2peaklist(dMass=(92.29387664794922, 94.06515...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.133, 260.149, 344.17, 374.18, 406.207, 43...","[183196.031, 1385752.875, 11007.416, 5626.116,...","[6.443324806529002, 48.73935219265959, 0.38715..."
4,14284,820.432617,820.432617,2,ok,1639.857394,H3N4,"{'F': 0, 'H': '3', 'N': '4', 'S': 0, 'G': 0, '...",14318,"MS2peaklist(dMass=(94.06523895263672, 95.04909...","[(1639.854, '(0, 3, 4, 0, 0, 0)')]","[246.134, 260.149, 344.171, 374.182, 432.222, ...","[23590.367, 2270437.5, 7480.896, 4565.527, 217...","[1.039022963635863, 100.0, 0.32949138657197125..."


In [15]:
from collections import namedtuple
import numpy as np
import ast
from decimal import *

def deftheoreticalms1(dataframe):
    #should be an element class holding these values
    df = dataframe#pd.read_csv("revised_zfNGbrain_20221129_1_updatedlist_parsed.csv", sep='\t', index_col=[0])
    #askopenfile?
    theomasslist = []
    newdf = pd.DataFrame()
    #print(df.dtypes)
    getcontext().prec = 8
    H = Decimal(1.007825) 
    C = Decimal(12.000)
    O = Decimal(15.9994)
    #N = 14
    Nadif = Decimal(22.989769) - Decimal(1.007825)   #replace H with Na
    mperMedHex = Decimal(174.08921)
    mperMeHex = Decimal(204.09977)
    mperMeHexNAc = Decimal(245.12632)
    mperMeNeu5Ac = Decimal(361.17367)
    mperMeNeu5Gc = Decimal(391.18423)
    mperMeKDN = Decimal(335.1700)
    mperMefreeend = Decimal(46.04186)  #CH3COH3
    mperMeterminal = Decimal(15.023475) #CH3, in positive mode (H+)
    #read the df
    #for testing: obsmass = {'F': 1, 'H':3, 'N': 2, 'S': 0, 'G': 0 }
    #get the structure dict
    for i in range(len(df)):
        tmp = df.iloc[i]['Structure dict']
        obsmass = ast.literal_eval(tmp)
        theomass = 0
        #calculate it
        #print(obsmass, obsmass['F'], type(obsmass['F']))
        Fuc = Decimal(obsmass['F'])
        Hex = Decimal(obsmass['H'])
        HexNAc = Decimal(obsmass['N'])
        Ac = Decimal(obsmass['S'])
        Gc = Decimal(obsmass['G'])
        KDN = Decimal(obsmass['KDN'])
        keys = list(obsmass)
        #print('keys', keys[-1])
        if (keys[-1] == 'Na') or (keys[-2] == 'Na'):
            #print('find Na')
            sodium = 1
        else:
            sodium = 0
        if(keys[-1] == 'Questionable'):
            print('it has a mass difference from theoretical value')
        theomass = str((Fuc * mperMedHex) + (Hex * mperMeHex) + (HexNAc * mperMeHexNAc) + (Ac * mperMeNeu5Ac) + (Gc * mperMeNeu5Gc) + (KDN * mperMeKDN)+ mperMefreeend + H + (Nadif * sodium))
        theomasslist.append(theomass)
    print(theomasslist)
    df.insert(5, 'Theoretical mass' , theomasslist, allow_duplicates = True)
    print(df)
    df.to_csv('Annotated_revised_zfNGintenstine_20230612_cloud.csv', sep='\t', index= False)
    print('finish writing csv')
    #add a new column and insert

In [16]:
deftheoreticalms1(res2)

['1639.8543', '1639.8543', '1639.8543', '1639.8543', '1639.8543', '1639.8543', '1639.8543', '1639.8543', '1639.8543', '1639.8543', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1813.9435', '1843.9541', '1884.9806', '1884.9806', '1884.9806', '1884.9806', '1884.9806', '1884.9806', '1884.9806', '1884.9806', '2018.0433', '2048.0539', '2048.0539', '2048.0539', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '2059.0698', '1568.8172', '1568.8172', '1568.8172', '1568.8172', '2222.1431', '2222.1431', '2222.1431', '2222.1431', '2222.1431', '2222.1431', '2222.1431', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '2263.1696', '1755.9017', '1755.9017', '2467.2694', '2467.2694', '2467.2694', '2495.2642', '2495.2642', '2495.2642'

In [20]:
try:
    theoglycotopedict = pd.read_csv("glycotope_deducer.csv")
except:
    theoglycotopedict = {}

In [14]:
#hand writing rules of glycotopes
glycotopeheader = ["Name of glycotope feature", "used units", "enzyme list and presence 0,1 or ?", "ion list"]
NG_core= ["NG core", {"H":3, "N":2}, {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "unknown enzyme": '?'}, [1234.56, 888.88]]
Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"MGAT3": 1}, [1234.56, 888.88]]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {"MGAT4":1 ,"MGAT5":1}, [1234.56, 888.88]]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {"B4GALT1":1, "B4TALT2~7": 1}, [1234.56, 888.88]] #re?
glycoenzymelist = [glycotopeheader,NG_core,Bisecting_NG_core,NG_elongationGal]
tmp = []
for glycoepitopes in glycoenzymelist:
    tmp.append(glycoepitopes)
print(f"tmp is+ {tmp}")
c = pd.DataFrame(tmp)
c.columns = c.iloc[0]
c = c[1:]
#set first row as index column, copied from https://www.statology.org/pandas-set-first-row-as-header/
c

tmp is+ [['Name of glycotope feature', 'used units', 'enzyme list and presence 0,1 or ?', 'ion list'], ['NG core', {'H': 3, 'N': 2}, {'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}, [1234.56, 888.88]], ['Bisecting NG core', {'H': 3, 'N': 3}, {'MGAT3': 1}, [1234.56, 888.88]], ['First and elongation of Galactose at NG ', {'H': 1}, {'B4GALT1': 1, 'B4TALT2~7': 1}, [1234.56, 888.88]]]


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]"
2,Bisecting NG core,"{'H': 3, 'N': 3}",{'MGAT3': 1},"[1234.56, 888.88]"
3,First and elongation of Galactose at NG,{'H': 1},"{'B4GALT1': 1, 'B4TALT2~7': 1}","[1234.56, 888.88]"
